# Large Language Model from Scratch

This notebook demonstrates how to create a Large Language Model from scratch, based on the book **"Large Language Model from Scratch"** by **Sebastian Raschka**.

## Purpose

The purpose of this code is to demonstrate the step-by-step process of building a language model. We will cover:
- Tokenization and vocabulary creation
- Data loading and preprocessing
- Implementing a Transformer-based architecture
- Training the model on a dataset
- Generating text using the trained model

This educational project aims to provide a deep understanding of the core concepts behind modern language models.

## Step 1: Install Requirements

This cell verifies that all required packages are installed and available. The following packages are required:
- `torch` – PyTorch for deep learning
- `tiktoken` – Tokenization library used by OpenAI models
- `tqdm` – Progress bar library
- `requests` – HTTP library for downloading datasets

**Prerequisites:**
1. Virtual environment is activated (`.venv`)
2. Requirements are installed via `uv pip install -r requirements.txt`

If you haven't set up the environment yet, run these commands in your terminal:
```bash
uv venv .venv
source .venv/bin/activate
uv pip install -r requirements.txt
```

In [ ]:
# Verify that all required packages are installed
import torch
import tiktoken
import tqdm
import requests

print(f"PyTorch version: {torch.__version__}")
print(f"tiktoken version: {tiktoken.__version__}")
print(f"tqdm version: {tqdm.__version__}")
print(f"requests version: {requests.__version__}")
print("\nAll required packages are installed and ready!")

## Step 2: Download the Dataset

This cell downloads the dataset file `the-verdict.txt` from GitHub. The file contains a short story titled "The Verdict" by Kate Chopin, which will be used as the training data for our language model.

The dataset is a plain text file that we will later tokenize and feed into our model for training.

In [ ]:
import requests
import os

# URL of the dataset
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

# Destination file path
file_path = "the-verdict.txt"

# Download the file
try:
    response = requests.get(url)
    response.raise_for_status()    # Raise an exception for bad status codes
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(response.text)
    print(f"Successfully downloaded '{file_path}'")
except requests.exceptions.RequestException as e:
    print(f"Error downloading file: {e}")

# Verify the file exists and show its size
if os.path.exists(file_path):
    file_size = os.path.getsize(file_path)
    print(f"File exists. Size: {file_size} bytes")
else:
    print("File not found!")

## Step 3: Device Selection (GPU/CPU)

Before we start training, we need to determine which device to use for computation. PyTorch supports multiple backends:

1. **CUDA** - NVIDIA GPUs (fastest, requires NVIDIA GPU + CUDA installed)
2. **MPS** - Apple Silicon (M1/M2/M3 chips, optimized for Mac)
3. **CPU** - Fallback option if no GPU is available

This cell will automatically detect the best available device and set it up for training.

In [ ]:
import torch

# Determine the device to use
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("CUDA (NVIDIA GPU) is available. Using GPU for training.")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("MPS (Apple Silicon) is available. Using GPU for training.")
    print(f"MPS device: {torch.device('mps')}")
else:
    device = torch.device('cpu')
    print("No GPU available. Using CPU for training (this will be slower).")

print(f"\nSelected device: {device}")

## Step 4: Tokenization with tiktoken

Now that we have our dataset and device set up, we need to convert the raw text into tokens that the language model can process. This process is called **tokenization**.

### What is tiktoken?

`tiktoken` is a fast BPE (Byte Pair Encoding) tokenizer library developed by OpenAI. It's used by GPT-3, GPT-4, and other OpenAI models.

**Benefits of tiktoken:**
- Extremely fast tokenization (written in Rust)
- Supports multiple tokenization schemes (cl100k_base, p50k_base, etc.)
- Handles unknown words gracefully with byte-level encoding
- Industry standard for OpenAI-style models

In this cell, we will:
1. Read the downloaded text file
2. Initialize the tiktoken tokenizer
3. Encode the text into token IDs
4. Verify the tokenization works correctly

In [ ]:
import tiktoken
import os

# Read the text file
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Total characters in text: {len(raw_text)}")
print(f"First 200 characters:\n{raw_text[:200]}\n")

# Initialize the tokenizer
# 'cl100k_base' is the encoding used by GPT-3, GPT-4, and text-embedding-ada-002
enc = tiktoken.get_encoding("cl100k_base")

# Encode the text into token IDs
tokens = enc.encode(raw_text)
print(f"Total tokens: {len(tokens)}")
print(f"First 50 token IDs: {tokens[:50]}")

# Decode a sample to verify
sample_tokens = tokens[:100]
decoded_text = enc.decode(sample_tokens)
print(f"\nDecoded sample:\n{decoded_text}")

print(f"\nVocabulary size: {enc.n_vocab} tokens")

## Step 5: Create PyTorch Dataset and DataLoader

Now we need to prepare our data for training. PyTorch provides `Dataset` and `DataLoader` classes that make it easy to:
- Load data in batches
- Shuffle data for training
- Handle large datasets efficiently with memory mapping

### What we'll do:
1. Create a custom `GPTDatasetV1` class that inherits from `torch.utils.data.Dataset`
2. The dataset will create input-target pairs for next-token prediction
3. Use `DataLoader` to create batches of data for training

### How it works:
- For each position in the text, the model tries to predict the next token
- Input: tokens at positions [i, i+1, ..., i+context_length-1]
- Target: token at position [i+context_length]
- This creates a sliding window over the text

In [ ]:
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    """A dataset for next-token prediction in GPT"""
    
    def __init__(self, tokens, tokenizer, block_size):
        """
        Args:
            tokens: List of token IDs from the text
            tokenizer: The tokenizer used to encode/decode
            block_size: Size of the input context (how many tokens to look back)
        """
        self.tokens = tokens
        self.tokenizer = tokenizer
        self.block_size = block_size
    
    def __len__(self):
        """Return the number of samples in the dataset"""
        # We can't predict beyond the end, so subtract block_size
        return len(self.tokens) - self.block_size
    
    def __getitem__(self, idx):
        """Return input and target for a given index"""
        # Get the input context (block_size tokens)
        input_chunk = self.tokens[idx:idx + self.block_size]
        # Get the target (next token after the input)
        target_chunk = self.tokens[idx + 1:idx + self.block_size + 1]
        
        # Convert to tensors
        return torch.tensor(input_chunk, dtype=torch.long), \
               torch.tensor(target_chunk, dtype=torch.long)

# Parameters
BLOCK_SIZE = 64  # Context window size
BATCH_SIZE = 4   # Number of samples per batch

# Create the dataset
dataset = GPTDatasetV1(
    tokens=tokens,
    tokenizer=enc,
    block_size=BLOCK_SIZE
)

# Create the DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0  # Set to >0 for multi-processing (use 0 on Windows)
)

# Print dataset info
print(f"Dataset size: {len(dataset)} samples")
print(f"Batch size: {BATCH_SIZE}")
print(f"Block size (context window): {BLOCK_SIZE}")
print(f"Number of batches: {len(dataloader)}\n")

# Get one batch to inspect
input_batch, target_batch = next(iter(dataloader))
print(f"Input batch shape: {input_batch.shape}")
print(f"Target batch shape: {target_batch.shape}")
print(f"\nFirst sample input: {input_batch[0].tolist()}")
print(f"First sample target: {target_batch[0].tolist()}")

# Decode the first sample to see what it looks like
first_input = enc.decode(input_batch[0].tolist())
first_target = enc.decode(target_batch[0].tolist())
print(f"\nFirst sample input text:\n{first_input}")
print(f"\nFirst sample target text:\n{first_target}")